In [1]:
from pathlib import Path

import contextily as cx
import geopandas as gpd
import matplotlib as mpl
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import shapely

# matplotlib's Agg renderer simplifies complex paths for performance by default (`path.simplify`),
# which visibly rounds off/cuts corners on the CSD polygons (Toronto's boundary alone has ~116k
# vertices) even though the underlying geometry is never touched. Disabling it here, not
# `gdf.simplify(...)` anywhere on the data - there's no actual simplification of the geometry in
# this notebook.
mpl.rcParams["path.simplify"] = False
mpl.rcParams["path.simplify_threshold"] = 0

In [2]:
GEOMS_DIR = Path("../../data/final_geoms")
OUT_DIR = Path("../../static/match-activity/zoi")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CITIES = {
    # Toronto's CSD boundary doesn't line up perfectly with the water layer around the islands -
    # `layered_land` swaps in a base of all GTHA CSDs (as a land backdrop) plus a second water
    # pass on top, so any seam shows as a sensible land/water color instead of a stray gap.
    "toronto": {"csd_src": GEOMS_DIR / "csd_gtha/csds_metro_gtha.shp", "csd_name": "Toronto", "layered_land": True},
    "vancouver": {"csd_src": GEOMS_DIR / "csd_vancouver/csds_metro_van.shp", "csd_name": "Vancouver", "layered_land": False},
}

# Fan zone & stadium point locations, EPSG:4326 (lon, lat)
POINTS = {
    "toronto": {"fanzone": (-79.404883, 43.638728), "stadium": (-79.418571, 43.633209)},
    "vancouver": {"fanzone": (-123.040699, 49.283063), "stadium": (-123.111977, 49.276693)},
}

LAND_COLOR = "#548581"
WATER_COLOR = "#586275"
CSD_COLOR = "#f1c500"
CSD_BORDER_COLOR = "#fff6d6"  # slightly yellow-white, just enough to separate the CSD from the islands/water around it
CSD_BORDER_WIDTH = 0  # the Toronto Islands are small slivers - anything much thicker reads as border, not fill
BUFFER_FILL_COLOR = "#e2941f"  # darker orange-red - the previous lighter orange washed out against the yellow CSD
BUFFER_EDGE_COLOR = "#b5970f"
BUFFER_ALPHA = 0.92
POINT_COLOR = "#dc4633"  # fan zone & stadium markers - the brand red
POINT_EDGE_COLOR = "#ffffff"
POINT_EDGE_WIDTH = 1.3
POINT_SIZE = 45  # matplotlib scatter `s` (marker area in points^2)

BASEMAP_URL = "https://services.arcgisonline.com/arcgis/rest/services/Elevation/World_Hillshade/MapServer/tile/{z}/{y}/{x}"
# Hillshade tiles are near-white over flat terrain, so blending them in at a flat `alpha` (the
# previous approach) washes the land color out toward white instead of shading it. Applied as a
# darken-only overlay instead (alpha scales with how dark the tile pixel is), so flat/white areas
# leave the land color untouched and only real relief darkens it.
SHADE_STRENGTH = 0.65

FIGSIZE = (5, 5)
DPI = 250
BOUNDS_PAD_FRAC = 0.08  # padding around the CSD extent, as a fraction of its width/height

## Load layers

`load_city_layers` reproduces the CSD workaround from `process_geometry.ipynb` (filter the
GTHA/Metro Vancouver collections by name rather than trust the broken standalone
`toronto_csd.shp`), fixes Toronto's stray `Z` dimension and Vancouver's un-dissolved 2km buffer,
and reprojects everything to EPSG:3857 to match the water layer and the basemap tiles. For
cities with `layered_land` set, it also keeps the *unfiltered* CSD collection around (all_csds)
to use as a land backdrop. It also builds a small point layer (`pts`) from `POINTS` for the fan
zone & stadium markers.

In [3]:
def load_city_layers(city, cfg):
    csd_all = gpd.read_file(cfg["csd_src"])
    csd = csd_all[csd_all["CSDNAME"] == cfg["csd_name"]].copy()
    assert len(csd) == 1, f"expected exactly 1 CSD feature for {cfg['csd_name']!r}, got {len(csd)}"
    csd = csd.to_crs(3857)

    water = gpd.read_file(GEOMS_DIR / f"{city}/{city}_water.gpkg").to_crs(3857)

    buf = gpd.read_file(GEOMS_DIR / f"{city}/{city}_merged_2km.shp")
    buf = buf.set_geometry(shapely.force_2d(buf.geometry))  # drop Toronto's stray Z=0 dimension
    if len(buf) > 1:  # Vancouver's file has 2 separate rows, not dissolved
        dissolved = buf.union_all() if hasattr(buf, "union_all") else buf.unary_union
        buf = gpd.GeoDataFrame({"id": [0]}, geometry=[dissolved], crs=buf.crs)
    buf = buf.to_crs(3857)

    all_csds = csd_all.to_crs(3857) if cfg.get("layered_land") else None

    city_points = POINTS[city]
    pts = gpd.GeoDataFrame(
        {"label": list(city_points.keys())},
        geometry=gpd.points_from_xy(*zip(*city_points.values())),
        crs=4326,
    ).to_crs(3857)

    return {"csd": csd, "water": water, "buf": buf, "all_csds": all_csds, "pts": pts}


layers = {city: load_city_layers(city, cfg) for city, cfg in CITIES.items()}
for city, l in layers.items():
    n_all = len(l["all_csds"]) if l["all_csds"] is not None else 0
    print(f"{city}: csd bounds={l['csd'].total_bounds}, water features={len(l['water'])}, buffer features={len(l['buf'])}, all_csds features={n_all}, points={list(l['pts']['label'])}")

toronto: csd bounds=[-8865406.59092569  5400551.84254289 -8807078.57704142  5443102.31176159], water features=2177, buffer features=1, all_csds features=27, points=['fanzone', 'stadium']
vancouver: csd bounds=[-13717333.02720819   6308841.24288651 -13694850.64998168
   6328322.97997259], water features=1027, buffer features=1, all_csds features=0, points=['fanzone', 'stadium']


## Render & save

Extent: each city's own CSD bounding box, padded and squared off, but then both cities' squares
are widened to share the same side length (the larger of the two) - otherwise Toronto and
Vancouver render at different physical scales (Vancouver's CSD is much smaller in km than
Toronto's, so fitting each to its own bbox zooms Vancouver in far more) and the 2km buffer
wouldn't be comparable between the two maps.

Two layer orders, depending on `layered_land`:

- **Vancouver** (unchanged): land rectangle, darken-only hillshade overlay, CSD fill, water,
  buffer.
- **Toronto** (`layered_land`): its CSD boundary doesn't line up perfectly with the water layer
  around the islands, so instead of a flat land rectangle it gets a water-colored base, then
  *all* GTHA CSDs filled in as a land backdrop (no borders), then the hillshade, then Toronto's
  own CSD highlighted in yellow, then the water layer a second time on top of that (so any seam
  between Toronto's polygon and the water dataset reads as a sensible land or water color instead
  of a stray gap), then the buffer.

The buffer is always the topmost layer in both orders - it should read over water, not get cut
off by it, since the 2km radius is measured from the stadium regardless of what's underneath.
The fan zone & stadium points are drawn last of all, above the buffer, as small red dots.

Either way the CSD fill sits on top of the hillshade (so it reads as a flat, opaque color with
no wash), and it gets a thin light border to separate it from the water/islands around it.

In [4]:
def square_bounds(minx, miny, maxx, maxy, pad_frac):
    w, h = maxx - minx, maxy - miny
    side = max(w, h) * (1 + 2 * pad_frac)
    cx_, cy_ = (minx + maxx) / 2, (miny + maxy) / 2
    return cx_, cy_, side


centers = {city: square_bounds(*l["csd"].total_bounds, BOUNDS_PAD_FRAC) for city, l in layers.items()}
shared_side = max(side for _, _, side in centers.values())
print(f"shared render window: {shared_side / 1000:.1f} km wide, for both cities")


def darken_overlay(xmin, ymin, xmax, ymax):
    """A black, variable-alpha image the same size as the hillshade tile mosaic - alpha is high
    where the tile is dark (relief) and ~0 where it's light (flat), so compositing it only ever
    darkens the land color underneath, never lightens it."""
    img, ext = cx.bounds2img(xmin, ymin, xmax, ymax, source=BASEMAP_URL, ll=False)
    gray = img[..., :3].astype(float).mean(axis=-1) / 255
    overlay = np.zeros((*gray.shape, 4))
    overlay[..., 3] = (1 - gray) * SHADE_STRENGTH
    return overlay, ext


for city, l in layers.items():
    csd, water, buf, all_csds, pts = l["csd"], l["water"], l["buf"], l["all_csds"], l["pts"]

    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
    fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
    ax.set_position([0, 0, 1, 1])

    cx_, cy_, _ = centers[city]
    half = shared_side / 2
    xmin, ymin, xmax, ymax = cx_ - half, cy_ - half, cx_ + half, cy_ + half
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)

    if all_csds is not None:
        # base = water color, so any gap between the CSD/land layers and the real water layer
        # reads as water rather than a stray background color
        ax.add_patch(mpatches.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, facecolor=WATER_COLOR, edgecolor="none", zorder=0))
        all_csds.plot(ax=ax, facecolor=LAND_COLOR, edgecolor="none", zorder=1)
        overlay, ext = darken_overlay(xmin, ymin, xmax, ymax)
        ax.imshow(overlay, extent=ext, zorder=2, interpolation="bilinear")
        csd.plot(ax=ax, facecolor=CSD_COLOR, edgecolor=CSD_BORDER_COLOR, linewidth=CSD_BORDER_WIDTH, zorder=3)
        water.plot(ax=ax, color=WATER_COLOR, lw=0, zorder=4)  # drawn again, to paper over any seam
        buf.plot(ax=ax, facecolor=BUFFER_FILL_COLOR, edgecolor=BUFFER_EDGE_COLOR, alpha=BUFFER_ALPHA, linewidth=1.2, zorder=5)  # on top of the water repass, so it isn't cut off where it overlaps water
    else:
        ax.add_patch(mpatches.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, facecolor=LAND_COLOR, edgecolor="none", zorder=0))
        overlay, ext = darken_overlay(xmin, ymin, xmax, ymax)
        ax.imshow(overlay, extent=ext, zorder=1, interpolation="bilinear")
        csd.plot(ax=ax, facecolor=CSD_COLOR, edgecolor=CSD_BORDER_COLOR, linewidth=CSD_BORDER_WIDTH, zorder=2)
        water.plot(ax=ax, color=WATER_COLOR, lw=0, zorder=3)
        buf.plot(ax=ax, facecolor=BUFFER_FILL_COLOR, edgecolor=BUFFER_EDGE_COLOR, alpha=BUFFER_ALPHA, linewidth=1.2, zorder=4)

    ax.scatter(pts.geometry.x, pts.geometry.y, s=POINT_SIZE, color=POINT_COLOR, edgecolor=POINT_EDGE_COLOR, linewidth=POINT_EDGE_WIDTH, zorder=10)

    ax.set_xlim(xmin, xmax)  # imshow(extent=...) can nudge the axes' view - pin it back down
    ax.set_ylim(ymin, ymax)
    ax.margins(0, 0)
    ax.set_axis_off()

    f_out = OUT_DIR / f"{city}.png"
    fig.savefig(f_out)
    plt.close(fig)
    print(f"saved {f_out}")

shared render window: 67.7 km wide, for both cities


saved ../../static/match-activity/zoi/toronto.png


saved ../../static/match-activity/zoi/vancouver.png
